[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_D.ipynb)


# Set D — From Passive → Spiking, Adaptation & Bursting — LEGO–Colab
**Author: Neural Engineering Laboratory, University of Missouri - Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

A short passive recap, then spiking (threshold, f–I, refractory), **adaptation** surrogate, and **bursting‑like** behavior.

---
## Table of Contents
- [D0 Passive recap](#d0)
- [D1 Add HH & threshold search](#d1)
- [D2 gNa/gK toggles](#d2)
- [D3 Refractory (paired pulses)](#d3)
- [D4 f–I curve & rheobase](#d4)
- [D5 Waveform landmarks](#d5)
- [D6 Propagation teaser](#d6)
- [D7 Spike‑frequency adaptation (surrogate)](#d7)
- [D8 Quantify adaptation](#d8)
- [D9 Bursting‑like bouts](#d9)
- [D10 AHP/ADP notes](#d10)
- [D11 Phase‑plane glimpse](#d11)
- [D12 Numerical tips](#d12)
---


In [ ]:

!pip -q install neuron==8.2.4 >/dev/null 2>&1
try:
    from neuron import h
except Exception:
    from neuron import h
from neuron.units import ms, mV
import numpy as np, matplotlib.pyplot as plt
print('Starter ready (NEURON expected).')


## D0 — Passive recap <a id='d0'></a>

In [ ]:

# Passive recap: step response and τ
soma=h.Section(name='somaP'); soma.L=soma.diam=20; soma.Ra=100; soma.insert('pas'); soma.g_pas=0.0002; soma.e_pas=-65
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=40; stim.amp=-0.05
v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
h.finitialize(-65*mV); h.continuerun(60*ms)
plt.figure(figsize=(7,3.3)); plt.plot(t,v,'k'); plt.title('D0: Passive recap (τ via 63% rule)'); plt.tight_layout(); plt.show()


## D1 — Add HH & threshold search <a id='d1'></a>

In [ ]:

# D1 Add HH & threshold search
soma=h.Section(name='soma'); soma.L=soma.diam=20; soma.Ra=100; soma.insert('pas'); soma.insert('hh')
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=20
for A in [0.03,0.05,0.07,0.09]:
    stim.amp=A
    v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
    h.finitialize(-65*mV); h.continuerun(35*ms)
    plt.figure(figsize=(6.4,2.4)); plt.plot(t,v,'k'); plt.title(f'D1: I={A} nA'); plt.tight_layout(); plt.show()


## D4 — f–I curve & rheobase <a id='d4'></a>

In [ ]:

# D4 f–I curve (rough)
soma=h.Section(name='somaFI'); soma.L=soma.diam=20; soma.insert('hh')
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=300
amps=np.linspace(0.04,0.15,8); rates=[]
for A in amps:
    stim.amp=A
    v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
    h.finitialize(-65*mV); h.continuerun(320*ms)
    import numpy as np
    vv=np.array(v); cross=(vv[1:]>=0)&(vv[:-1]<0); spikes=cross.sum(); rates.append(1000*spikes/300.0)
plt.figure(figsize=(5.4,3.2)); plt.plot(amps,rates,'o-'); plt.xlabel('nA'); plt.ylabel('Hz'); plt.title('D4: f–I'); plt.tight_layout(); plt.show()


## D7 — Spike‑frequency adaptation (surrogate) <a id='d7'></a>

In [ ]:

# D7 Adaptation surrogate via slow inhibitory conductance
soma=h.Section(name='somaSFA'); soma.L=soma.diam=20; soma.insert('hh')
stim=h.IClamp(soma(0.5)); stim.delay=5; stim.dur=1000; stim.amp=0.12
synI=h.AlphaSynapse(soma(0.5)); synI.onset=100; synI.tau=200; synI.gmax=0.002; synI.e=-70
v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
h.finitialize(-65*mV); h.continuerun(1100*ms)
plt.figure(figsize=(7,3.3)); plt.plot(t,v,'k'); plt.title('D7: Adaptation surrogate'); plt.tight_layout(); plt.show()


## D9 — Bursting‑like bouts <a id='d9'></a>

In [ ]:

# D9 Bursting-like bouts via slow envelope
soma=h.Section(name='somaBurst'); soma.L=soma.diam=20; soma.insert('hh')
ic=h.IClamp(soma(0.5)); ic.delay=0; ic.dur=1e9; ic.amp=0.0
T=4000; import numpy as np
tt=np.arange(0,T,0.1); envelope=0.07+0.05*np.sin(2*np.pi*tt/1500.0)
ivec=h.Vector(envelope); tvec=h.Vector(tt)
ivec.play(ic._ref_amp,tvec,1)
v=h.Vector().record(soma(0.5)._ref_v); t=h.Vector().record(h._ref_t)
h.finitialize(-65*mV); h.continuerun(T*ms)
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3.3)); plt.plot(t,v,'k'); plt.title('D9: Bursting-like bouts'); plt.tight_layout(); plt.show()


## Reflection
- How do the passive τ expectations shape spike latency and f–I slope?
- Which adaptation/bursting parameters best serve as **control knobs** for networks in later sets?


---

## Practice / Discussion Questions — Set D — Spiking, Adaptation, Bursting (20)

1) State how voltage‑gated Na⁺ and K⁺ currents produce a spike’s upstroke and downstroke.
2) *Explain*: What is **absolute vs relative refractoriness** at the channel level?
3) Describe one mechanism for **spike‑frequency adaptation** and its effect on ISIs.
4) Define **bursting** and name a minimal set of variables that can support it.
5) When is a **reduced model** preferable to HH‑type dynamics for teaching/exploration?
6) *Predict*: How will adding a slow outward current change the ISI sequence to a step input?
7) Describe the **phase‑plane (qualitative)** idea behind spike initiation in a 2‑variable model.
8) Compare two **adaptation** mechanisms (e.g., M‑current vs Ca²⁺‑dependent K⁺).
9) *Critique*: One pitfall of teaching with an overly simple spike model.
10) Explain how spike threshold can **vary** with recent membrane history.
11) Why might a neuron fail to spike to inputs that spiked it earlier (without parameter changes)?
12) *Scenario*: Two cells receive identical barrages; one bursts, one adapts. Suggest a minimal ionic difference.
13) A good **metric** to quantify adaptation in a step‑depolarization protocol.
14) One reason to add noise to spike models and one reason not to (for education).
15) Outline a **spike‑count vs input** experiment and what would indicate adaptation.
16) Explain “**dynamic threshold**” in intuitive terms.
17) *Design*: Two‑condition step protocol to elicit bursting and list expected traces.
18) Two **biophysical levers** that can turn adapting into bursting.
19) How does adding adaptation change a neuron’s role in a **network**?
20) Summarize how Set D concepts inform **STM** and **WTA** behavior.
